# Project 2

Welcome to the second NLP1 project. The goal of the second tutorial was to get you familiar with using pre-trained embeddings and moving towards a compositional encoder for text classification.

Given the skills you have developed in the last week, you will now be applying them within the dataset and task you have chosen during Project 1. You will be applying hands-on the experiments you have already explored in the Tutorial 2, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and errors analyses, and tackling your own research questions. 


The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

# Get to work!

1. Find a word embedding model that is appropriate for your dataset. Think about the language, the kind of text you are working with. Find evidence that your choice is a good one: 
    * Can you tokenize your texts in a way coherent with the definition of a token in the embedding model?
    * Does the model's vocabulary give enough coverage to the words in your texts? etc.
2. Explore what your chosen word embedding model captures. Some things you might consider: 
    * Is it good at representing interesting similarities (re to your choice of task)? J
    * Have you found task-specific evidence that it's better than some other reasonable alternative?
    * Do you know about limitations of this model?
    * Any limitations that you find particularly troubling for your task? 
3. Experiment with the embed-and-pool approach.  
    * What design choices have meaningful impact in classification performance (eg, embedding size, architecture design, optimisation parameters), if any?
    * Is this better than what you could achieve last week? In other words, did pretrained embeddings add to your design?
4. Experiment with a recurrent encoder (GRU or LSTM). 
    * What design choices have meaningful impact in classification performance, if any?
    * Did you investigate the impact of data size, embedding parameters (trained or not, fixed or trainable, etc)?
    * Did you struggle with over-fitting?
    * How far can this approach go? Are you able to improve meaningfully on embed-and-pool?
5. We make decisions under uncertainty all the time, are you gathering enough information to substantiate your choices? Various things can lead to small differences in results, and these differences are not always consistent. Here are some things to think about:
    * How did you structure you hyperparameter search?
    * Are you keeping an eye on experimental variance?
6. Experiment with error analysis and try to discover what characteristics of input texts are predictive of performance differences across major design choices. 
    * What analysis features did you come up with?
    * Which ones did you manage to test and how did that go?
    * What would you prioritise in the future in order to improve your systems?

Be mindful that you will be using your experiments to write up your final blog, so make sure your code is clean and neat!

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

# Tips

## Pretrained embedding models

We will post a list with some options for you. 

## Features for analysis

Features that you can interpret as an aspect of _complexity_ in light of your task's goals.

- length
- presence of rare/common words, including unknown words
- syntactic features (like parts-of-speech, syntactic complexity, syntactic voice, presence of negation, etc.)
- semantic features (eg, the average similarity of the text to training documents or some curated set of training documents, presence of words in certain semantic classes, etc.)
- task-specific features (eg, 
- model-specific features (eg, predicted label, probability assigned to predicted label, number of parameters or layers or some other model descriptor)

# 1. Embedding choice: GloVe 50d vs fastText 300d

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

import nltk
from datasets import load_dataset

sys.path.insert(0, str(Path.cwd().parent.parent))
import part2

nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/efearas/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
sst2 = load_dataset("stanfordnlp/sst2")

In [ ]:
!wget -nc https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip
!unzip -n glove.6B.zip glove.6B.50d.txt

glove = part2.clslib.PretrainedEmbeddings("glove.6B.50d.txt", 50)

Bestand '‘glove.6B.zip’' is reeds aanwezig -- wordt niet opgehaald.

Archive:  glove.6B.zip


In [ ]:
!wget -nc https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip
fasttext = part2.clslib.PretrainedEmbeddings("wiki-news-300d-1M.vec.zip", 300, skip_first_line=True)    

Bestand '‘wiki-news-300d-1M.vec.zip’' is reeds aanwezig -- wordt niet opgehaald.



In [6]:
tokenizer = part2.clslib.BasicTokenizer(lowercase=True, remove_punct=False, remove_stopwords=False, language='english')
print(tokenizer.tokens(sst2["train"]["sentence"][1]))

for token in ["n't", "'s"]:
    print(token, token in glove, token in fasttext)

['contains', 'no', 'wit', ',', 'only', 'labored', 'gags']
n't True False
's True True


In [ ]:
from collections import Counter

word_counts = Counter(word for sentence in sst2["train"]["sentence"] for word in tokenizer.tokens(sentence))

for name, model in [("glove", glove), ("fasttext", fasttext)]:
    unknown = {word: count for word, count in word_counts.items() if word not in model}
    type_coverage = 1 - len(unknown) / len(word_counts)
    token_coverage = 1 - sum(unknown.values()) / sum(word_counts.values())
    print(f"{name}: type coverage {type_coverage:.1%}, token coverage {token_coverage:.1%}")
    print("most common unknown words:", Counter(unknown).most_common(15))


glove: type coverage 92.9%, token coverage 99.0%
most common unknown words: [('dumbed-down', 37), ('slo-mo', 23), ('picture-perfect', 22), ('stomach-turning', 21), ('ham-fisted', 21), ('mind-numbingly', 20), ('not-very-funny', 19), ('hour-and-a-half', 17), ('dime-store', 16), ('bargain-basement', 16), ('sequel-for-the-sake', 16), ('self-caricature', 16), ('paint-by-number', 16), ('still-inestimable', 15), ('pre-credit', 15)]
fasttext: type coverage 95.7%, token coverage 98.6%
most common unknown words: [("n't", 2311), ('``', 1757), ('`', 970), ("''", 43), ('and/or', 27), ('1/2', 25), ('writer/director', 23), ('clockstoppers', 22), ('not-very-funny', 19), ('sequel-for-the-sake', 16), ('still-inestimable', 15), ('nebrida', 15), ('fun-for-fun', 15), ('clung-to', 15), ('opera-ish', 15)]


# 2. What the embeddings capture: GloVe 50d vs fastText 300d

In [8]:
import numpy as np

def cos_similarity(u, v):
    return u @ v / (np.linalg.norm(u) * np.linalg.norm(v))

def topk_ids(v, E, k=10):
    sims = E @ v / (np.linalg.norm(E, axis=1) * np.linalg.norm(v) + 1e-8)
    return np.argsort(-sims)[:k]

In [9]:
for word in ["good", "bad", "great", "terrible", "boring", "funny", "masterpiece"]:
    ids = topk_ids(glove.embed_word(word), glove.vectors, k=11)[1:]
    print(word, [glove.word(i) for i in ids])

good ['better', 'really', 'always', 'sure', 'something', 'think', 'way', 'thing', 'little', 'very']
bad ['worse', 'unfortunately', 'too', 'really', 'little', 'bit', 'things', 'nothing', 'gone', 'trouble']
great ['greatest', 'good', 'perhaps', 'life', 'well', 'little', 'much', 'inspiration', 'luck', 'experience']
terrible ['horrible', 'awful', 'tragic', 'dreadful', 'tragedy', 'horrific', 'nightmare', 'unfortunate', 'horrendous', 'unfortunately']
boring ['awfully', 'pretty', 'funny', 'incredibly', 'silly', 'weird', 'terribly', 'stupid', 'bored', 'unbelievably']
funny ['hilarious', 'amusing', 'silly', 'joke', 'fun', 'weird', 'pretty', 'scary', 'boring', 'incredibly']
masterpiece ['masterwork', 'masterpieces', 'retelling', 'cinematic', 'fable', 'novel', 'renaissance', 'epic', 'fantasy', 'acclaimed']


In [10]:
for a, b in [("good", "bad"), ("good", "great"), ("great", "terrible"), ("funny", "boring")]:
    print(a, b, round(cos_similarity(glove.embed_word(a), glove.embed_word(b)), 2))

good bad 0.8
good great 0.8
great terrible 0.61
funny boring 0.81


In [11]:
nltk.download("opinion_lexicon")
from nltk.corpus import opinion_lexicon
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

pos = [w for w in opinion_lexicon.positive() if w in glove and w in fasttext]
neg = [w for w in opinion_lexicon.negative() if w in glove and w in fasttext]
words = pos + neg
labels = [1] * len(pos) + [0] * len(neg)
print("majority baseline:", round(len(neg) / len(words), 3))

for name, model in [("glove", glove), ("fasttext", fasttext)]:
    X = np.array([model.embed_word(w) for w in words])
    print(name, round(cross_val_score(LogisticRegression(max_iter=1000), X, labels, cv=5).mean(), 3))

[nltk_data] Downloading package opinion_lexicon to
[nltk_data]     /Users/efearas/nltk_data...
[nltk_data]   Package opinion_lexicon is already up-to-date!


majority baseline: 0.697
glove 0.882
fasttext 0.949


In [ ]:
# average cosine similarity within and between positive / negative words (pos, neg from the probe cell)
for name, model in [("glove", glove), ("fasttext", fasttext)]:
    P = np.array([model.embed_word(w) for w in pos])
    N = np.array([model.embed_word(w) for w in neg])
    P /= np.linalg.norm(P, axis=1, keepdims=True)
    N /= np.linalg.norm(N, axis=1, keepdims=True)
    pos_pos = ((P @ P.T).sum() - len(P)) / (len(P) ** 2 - len(P))  # leave out each word with itself
    neg_neg = ((N @ N.T).sum() - len(N)) / (len(N) ** 2 - len(N))
    pos_neg = (P @ N.T).mean()
    print(f"{name}: pos-pos {pos_pos:.3f}, neg-neg {neg_neg:.3f}, pos-neg {pos_neg:.3f}")

# 3. Embed-and-pool (GloVe 50d)

In [12]:
import part1

sst2_outcome_space = part1.clslib.NamedClasses(["negative", "positive"])
my_seeds = [0]
num_folds = 5
my_cross_validation_sets = []
for seed in my_seeds:
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            pairs=list(zip(sst2["train"]["sentence"], sst2["train"]["label"])),
            k=num_folds,
            seed=seed
        )
    )

In [ ]:
nltk.download("stopwords")
tokenizer_for_rnn = part2.clslib.BasicTokenizer(lowercase=True, remove_punct=False, remove_stopwords=False, language='english', vocab=glove.vocab)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/efearas/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
from collections import OrderedDict

my_classifiers = OrderedDict()
my_classifiers['embed-and-pool'] = []
for run, seed in enumerate(my_seeds):
    my_classifiers['embed-and-pool'].append(
        part2.experiment.cross_validation(
            outcome_space=sst2_outcome_space,
            data_splits=my_cross_validation_sets[run],
            tokenizer=tokenizer_for_rnn,
            emb_model=glove,
            enc_type='embed-and-pool',
            enc_kwargs={'freeze_embeddings': True},
            regressor_kwargs={'hidden_size1': 256, 'hidden_size2': 128, 'dropout': 0.3},
            opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 10},
            device='cpu',
            save_to=f"results/embed-and-pool/seed-{seed}",
        )
    )

In [ ]:
my_classifiers['embed-and-pool fine-tuned'] = []
for run, seed in enumerate(my_seeds):
    my_classifiers['embed-and-pool fine-tuned'].append(
        part2.experiment.cross_validation(
            outcome_space=sst2_outcome_space,
            data_splits=my_cross_validation_sets[run],
            tokenizer=tokenizer_for_rnn,
            emb_model=glove,
            enc_type='embed-and-pool',
            enc_kwargs={'freeze_embeddings': False},
            regressor_kwargs={'hidden_size1': 256, 'hidden_size2': 128, 'dropout': 0.3},
            opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 10},
            device='cpu',
            save_to=f"results/embed-and-pool-fine-tuned/seed-{seed}",
        )
    )

### Without pretrained embeddings (random init, trainable)

In [ ]:
import copy

random_emb = copy.deepcopy(glove)
random_emb.vectors = np.random.default_rng(0).normal(0, 0.1, glove.vectors.shape).astype(np.float32)
random_emb.vectors[0] = 0  # keep <pad> as zeros

my_classifiers['embed-and-pool random init'] = []
for run, seed in enumerate(my_seeds):
    my_classifiers['embed-and-pool random init'].append(
        part2.experiment.cross_validation(
            outcome_space=sst2_outcome_space,
            data_splits=my_cross_validation_sets[run],
            tokenizer=tokenizer_for_rnn,
            emb_model=random_emb,
            enc_type='embed-and-pool',
            enc_kwargs={'freeze_embeddings': False},
            regressor_kwargs={'hidden_size1': 256, 'hidden_size2': 128, 'dropout': 0.3},
            opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 10},
            device='cpu',
            save_to=f"results/embed-and-pool-random/seed-{seed}",
        )
    )

In [ ]:
my_classifiers['P1: TI + MLP2'] = [{'accuracies': {0: 0.9224, 1: 0.9203, 2: 0.9214, 3: 0.9209, 4: 0.9194}}]
part1.analysis.plot_box_and_points(part1.analysis.make_data_frame(my_classifiers))

# 4. LSTM (GloVe 50d)

In [17]:
my_classifiers['lstm'] = []
for run, seed in enumerate(my_seeds):
    my_classifiers['lstm'].append(
        part2.experiment.cross_validation(
            outcome_space=sst2_outcome_space,
            data_splits=my_cross_validation_sets[run],
            tokenizer=tokenizer_for_rnn,
            emb_model=glove,
            enc_type='lstm',
            enc_kwargs={'freeze_embeddings': True, 'hidden_size': 20, 'word_dropout': 0.},
            regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
            opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 8},
            device='cpu',
            save_to=f"results/lstm/seed-{seed}",
        )
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 7.818 seconds
Dataloader done! Time taken: 0.162 seconds


  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.4952 | train acc: 0.7557 | val loss: 0.4305 | val acc: 0.8014
Epoch 02 | train loss: 0.4235 | train acc: 0.8048 | val loss: 0.3976 | val acc: 0.8170
Epoch 03 | train loss: 0.3964 | train acc: 0.8195 | val loss: 0.3836 | val acc: 0.8261
Epoch 04 | train loss: 0.3793 | train acc: 0.8297 | val loss: 0.3754 | val acc: 0.8321
Epoch 05 | train loss: 0.3684 | train acc: 0.8372 | val loss: 0.3699 | val acc: 0.8356
Epoch 06 | train loss: 0.3582 | train acc: 0.8429 | val loss: 0.3629 | val acc: 0.8409
Epoch 07 | train loss: 0.3477 | train acc: 0.8467 | val loss: 0.3520 | val acc: 0.8434
Epoch 08 | train loss: 0.3399 | train acc: 0.8521 | val loss: 0.3535 | val acc: 0.8483
Training done! Time taken: 49.573 seconds
Tokenizing texts...
Tokenization done! Time taken: 7.314 seconds
Dataloader done! Time taken: 0.187 seconds


  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.4861 | train acc: 0.7621 | val loss: 0.4363 | val acc: 0.7967
Epoch 02 | train loss: 0.4188 | train acc: 0.8081 | val loss: 0.4092 | val acc: 0.8133
Epoch 03 | train loss: 0.3940 | train acc: 0.8229 | val loss: 0.3913 | val acc: 0.8261
Epoch 04 | train loss: 0.3771 | train acc: 0.8324 | val loss: 0.3840 | val acc: 0.8312
Epoch 05 | train loss: 0.3651 | train acc: 0.8402 | val loss: 0.3768 | val acc: 0.8383
Epoch 06 | train loss: 0.3519 | train acc: 0.8479 | val loss: 0.3638 | val acc: 0.8440
Epoch 07 | train loss: 0.3432 | train acc: 0.8535 | val loss: 0.3590 | val acc: 0.8458
Epoch 08 | train loss: 0.3334 | train acc: 0.8572 | val loss: 0.3627 | val acc: 0.8455
Training done! Time taken: 49.137 seconds
Tokenizing texts...
Tokenization done! Time taken: 7.261 seconds
Dataloader done! Time taken: 0.161 seconds


... (2 outputs left out)

  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.4961 | train acc: 0.7545 | val loss: 0.4338 | val acc: 0.7995
Epoch 02 | train loss: 0.4258 | train acc: 0.8030 | val loss: 0.4126 | val acc: 0.8097
Epoch 03 | train loss: 0.4016 | train acc: 0.8170 | val loss: 0.3913 | val acc: 0.8248
Epoch 04 | train loss: 0.3845 | train acc: 0.8277 | val loss: 0.3787 | val acc: 0.8333
Epoch 05 | train loss: 0.3694 | train acc: 0.8370 | val loss: 0.3644 | val acc: 0.8399
Epoch 06 | train loss: 0.3582 | train acc: 0.8418 | val loss: 0.3610 | val acc: 0.8424
Epoch 07 | train loss: 0.3484 | train acc: 0.8475 | val loss: 0.3514 | val acc: 0.8475
Epoch 08 | train loss: 0.3383 | train acc: 0.8545 | val loss: 0.3533 | val acc: 0.8487
Training done! Time taken: 48.916 seconds
Tokenizing texts...
Tokenization done! Time taken: 7.280 seconds
Dataloader done! Time taken: 0.258 seconds


  0%|          | 0/8 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.4915 | train acc: 0.7597 | val loss: 0.4326 | val acc: 0.7995
Epoch 02 | train loss: 0.4209 | train acc: 0.8057 | val loss: 0.4022 | val acc: 0.8149
Epoch 03 | train loss: 0.3969 | train acc: 0.8199 | val loss: 0.4031 | val acc: 0.8163
Epoch 04 | train loss: 0.3816 | train acc: 0.8285 | val loss: 0.3751 | val acc: 0.8353
Epoch 05 | train loss: 0.3675 | train acc: 0.8361 | val loss: 0.3686 | val acc: 0.8396
Epoch 06 | train loss: 0.3562 | train acc: 0.8441 | val loss: 0.3583 | val acc: 0.8438
Epoch 07 | train loss: 0.3474 | train acc: 0.8481 | val loss: 0.3679 | val acc: 0.8374
Epoch 08 | train loss: 0.3391 | train acc: 0.8518 | val loss: 0.3499 | val acc: 0.8493
Training done! Time taken: 49.007 seconds


In [19]:
my_classifiers['lstm fine-tuned'] = []
for run, seed in enumerate(my_seeds):
    my_classifiers['lstm fine-tuned'].append(
        part2.experiment.cross_validation(
            outcome_space=sst2_outcome_space,
            data_splits=my_cross_validation_sets[run],
            tokenizer=tokenizer_for_rnn,
            emb_model=glove,
            enc_type='lstm',
            enc_kwargs={'freeze_embeddings': False, 'hidden_size': 20, 'word_dropout': 0.},
            regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
            opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 5},
            device='cpu',
            save_to=f"results/lstm-fine-tuned/seed-{seed}",
        )
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 7.849 seconds
Dataloader done! Time taken: 0.170 seconds


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.3952 | train acc: 0.8221 | val loss: 0.2761 | val acc: 0.8877
Epoch 02 | train loss: 0.2331 | train acc: 0.9092 | val loss: 0.2375 | val acc: 0.9071
Epoch 03 | train loss: 0.1758 | train acc: 0.9350 | val loss: 0.2170 | val acc: 0.9174
Epoch 04 | train loss: 0.1461 | train acc: 0.9452 | val loss: 0.2079 | val acc: 0.9229
Epoch 05 | train loss: 0.1221 | train acc: 0.9550 | val loss: 0.2182 | val acc: 0.9254
Training done! Time taken: 263.521 seconds
Tokenizing texts...
Tokenization done! Time taken: 7.472 seconds
Dataloader done! Time taken: 0.161 seconds


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.3988 | train acc: 0.8202 | val loss: 0.3008 | val acc: 0.8759
Epoch 02 | train loss: 0.2369 | train acc: 0.9085 | val loss: 0.2455 | val acc: 0.9052
Epoch 03 | train loss: 0.1805 | train acc: 0.9329 | val loss: 0.2457 | val acc: 0.9103
Epoch 04 | train loss: 0.1508 | train acc: 0.9438 | val loss: 0.2427 | val acc: 0.9123
Epoch 05 | train loss: 0.1282 | train acc: 0.9526 | val loss: 0.2444 | val acc: 0.9180
Training done! Time taken: 258.363 seconds
Tokenizing texts...
Tokenization done! Time taken: 7.507 seconds
Dataloader done! Time taken: 0.178 seconds


... (2 outputs left out)

  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.3905 | train acc: 0.8256 | val loss: 0.2891 | val acc: 0.8841
Epoch 02 | train loss: 0.2295 | train acc: 0.9102 | val loss: 0.2441 | val acc: 0.9074
Epoch 03 | train loss: 0.1723 | train acc: 0.9351 | val loss: 0.2276 | val acc: 0.9161
Epoch 04 | train loss: 0.1429 | train acc: 0.9457 | val loss: 0.2346 | val acc: 0.9173
Epoch 05 | train loss: 0.1198 | train acc: 0.9551 | val loss: 0.2373 | val acc: 0.9189
Training done! Time taken: 266.274 seconds
Tokenizing texts...
Tokenization done! Time taken: 7.541 seconds
Dataloader done! Time taken: 0.162 seconds


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.4027 | train acc: 0.8137 | val loss: 0.2864 | val acc: 0.8869
Epoch 02 | train loss: 0.2351 | train acc: 0.9085 | val loss: 0.2415 | val acc: 0.9044
Epoch 03 | train loss: 0.1779 | train acc: 0.9345 | val loss: 0.2199 | val acc: 0.9160
Epoch 04 | train loss: 0.1458 | train acc: 0.9460 | val loss: 0.2395 | val acc: 0.9160
Epoch 05 | train loss: 0.1220 | train acc: 0.9543 | val loss: 0.2382 | val acc: 0.9214
Training done! Time taken: 267.424 seconds


In [20]:
small_sets = [{k: {'training': f['training'][:len(f['training']) // 10], 'validation': f['validation']} for k, f in splits.items()}
              for splits in my_cross_validation_sets]

my_classifiers['lstm 10% data'] = []
for run, seed in enumerate(my_seeds):
    my_classifiers['lstm 10% data'].append(
        part2.experiment.cross_validation(
            outcome_space=sst2_outcome_space,
            data_splits=small_sets[run],
            tokenizer=tokenizer_for_rnn,
            emb_model=glove,
            enc_type='lstm',
            enc_kwargs={'freeze_embeddings': True, 'hidden_size': 20, 'word_dropout': 0.},
            regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
            opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 5},
            device='cpu',
            save_to=f"results/lstm-10pct/seed-{seed}",
        )
    )

  0%|          | 0/5 [00:00<?, ?it/s]

Tokenizing texts...
Tokenization done! Time taken: 2.150 seconds
Dataloader done! Time taken: 0.047 seconds


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6488 | train acc: 0.6107 | val loss: 0.5539 | val acc: 0.7214
Epoch 02 | train loss: 0.5353 | train acc: 0.7412 | val loss: 0.4953 | val acc: 0.7683
Epoch 03 | train loss: 0.4968 | train acc: 0.7680 | val loss: 0.4778 | val acc: 0.7765
Epoch 04 | train loss: 0.4774 | train acc: 0.7754 | val loss: 0.4745 | val acc: 0.7815
Epoch 05 | train loss: 0.4619 | train acc: 0.7886 | val loss: 0.4669 | val acc: 0.7815
Training done! Time taken: 5.758 seconds
Tokenizing texts...
Tokenization done! Time taken: 2.094 seconds
Dataloader done! Time taken: 0.046 seconds


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6376 | train acc: 0.6170 | val loss: 0.5687 | val acc: 0.7146
Epoch 02 | train loss: 0.5400 | train acc: 0.7344 | val loss: 0.5378 | val acc: 0.7329
Epoch 03 | train loss: 0.5027 | train acc: 0.7605 | val loss: 0.4879 | val acc: 0.7676
Epoch 04 | train loss: 0.4790 | train acc: 0.7750 | val loss: 0.4849 | val acc: 0.7736
Epoch 05 | train loss: 0.4653 | train acc: 0.7793 | val loss: 0.4693 | val acc: 0.7800
Training done! Time taken: 5.437 seconds
Tokenizing texts...
Tokenization done! Time taken: 2.095 seconds
Dataloader done! Time taken: 0.046 seconds


... (2 outputs left out)

  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6411 | train acc: 0.6247 | val loss: 0.5604 | val acc: 0.7147
Epoch 02 | train loss: 0.5360 | train acc: 0.7373 | val loss: 0.5061 | val acc: 0.7560
Epoch 03 | train loss: 0.4971 | train acc: 0.7635 | val loss: 0.4888 | val acc: 0.7673
Epoch 04 | train loss: 0.4756 | train acc: 0.7795 | val loss: 0.4718 | val acc: 0.7776
Epoch 05 | train loss: 0.4591 | train acc: 0.7882 | val loss: 0.4642 | val acc: 0.7826
Training done! Time taken: 5.412 seconds
Tokenizing texts...
Tokenization done! Time taken: 2.093 seconds
Dataloader done! Time taken: 0.046 seconds


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6542 | train acc: 0.6091 | val loss: 0.5739 | val acc: 0.7020
Epoch 02 | train loss: 0.5402 | train acc: 0.7324 | val loss: 0.5194 | val acc: 0.7533
Epoch 03 | train loss: 0.4924 | train acc: 0.7723 | val loss: 0.4916 | val acc: 0.7639
Epoch 04 | train loss: 0.4772 | train acc: 0.7777 | val loss: 0.4725 | val acc: 0.7757
Epoch 05 | train loss: 0.4587 | train acc: 0.7918 | val loss: 0.4642 | val acc: 0.7798
Training done! Time taken: 5.402 seconds


In [ ]:
part1.analysis.plot_box_and_points(part1.analysis.make_data_frame(my_classifiers))

In [22]:
import json
json.dump({name: [run['accuracies'] for run in runs] for name, runs in my_classifiers.items()}, open("results/accuracies.json", "w"))

# 5. Uncertainty: Bayesian comparison with ROPE

In [ ]:
# helper from the part1 tutorial
def get_all_measurements_for(data, sys_name):
    return np.array([list(run_data['accuracies'].values()) for run_data in data[sys_name]])

pairs = [
    ('embed-and-pool', 'embed-and-pool fine-tuned'),
    ('embed-and-pool random init', 'embed-and-pool fine-tuned'),
    ('embed-and-pool', 'lstm'),
    ('lstm 10% data', 'lstm'),
    ('lstm', 'lstm fine-tuned'),
    ('embed-and-pool fine-tuned', 'lstm fine-tuned'),
    ('P1: TI + MLP2', 'lstm fine-tuned'),
]

for a, b in pairs:
    posterior = part1.analysis.BenavoliCorrelatedT(
        get_all_measurements_for(my_classifiers, a).flatten(),
        get_all_measurements_for(my_classifiers, b).flatten(),
        runs=len(my_seeds),
        names=[a, b]
    )
    p_a, p_tie, p_b = posterior.probs(0.005)
    print(f"A={a} B={b}: P(A wins)={p_a:.4f} P(Tie)={p_tie:.4f} P(B wins)={p_b:.4f} Effect Size={posterior.effect_size:.2f}")

# 6. Error analysis on SST-2 validation

In [ ]:
import pandas as pd

val_texts = sst2['validation']['sentence']
val_labels = np.array(sst2['validation']['label'])

negations = {"not", "n't", "no", "never", "nothing", "nobody", "none", "nor", "neither", "without"}
contrasts = {"but", "however", "although", "though", "yet"}

# features per validation sentence: length, negation, contrast, unknown words
val_tokens = [tokenizer_for_rnn.tokens(x) for x in val_texts]
val_features = [[len(t), int(any(w in negations for w in t)), int(any(w in contrasts for w in t)), sum(w not in glove for w in t)]
                for t in val_tokens]

# model name -> results folder
model_folders = {
    'embed-and-pool': 'embed-and-pool',
    'embed-and-pool fine-tuned': 'embed-and-pool-fine-tuned',
    'lstm': 'lstm',
    'lstm fine-tuned': 'lstm-fine-tuned',
}

table = []
systems = dict()
for model_family, folder in model_folders.items():
    systems[model_family] = len(systems)
    for fold in range(num_folds):
        trained_nn = part2.experiment.load(f"results/{folder}/seed-0/fold-{fold}/model", glove)
        logprobs = part2.experiment.predict_logprobs(trained_nn, tokenizer_for_rnn, val_texts, 64, 'cpu')
        preds = np.argmax(logprobs, -1)
        for i in range(len(preds)):
            table.append([model_family, systems[model_family], i, fold, val_labels[i], preds[i], int(val_labels[i] == preds[i])] + val_features[i])

df = pd.DataFrame(table, columns=['system', 'system_id', 'item_id', 'fold', 'label', 'pred', 'correct', 'length', 'negation', 'contrast', 'unknown'])
df.groupby('system')['correct'].mean()

In [ ]:
# accuracy with / without negation and contrast
print(df.groupby(['system', 'negation'])['correct'].mean().unstack())
print(df.groupby(['system', 'contrast'])['correct'].mean().unstack())

In [ ]:
import arviz as az

feature_names = [f"pred:{n}" for n in sst2_outcome_space.names] + ['length', 'negation', 'contrast', 'unknown']

mcmc = part2.analysis.fit_bda_model(
    num_systems=len(systems),
    sys_id=df['system_id'].to_numpy(),
    features=np.hstack([
        part2.analysis.one_hot(df['pred'].to_numpy(), len(sst2_outcome_space)),
        df[['length', 'negation', 'contrast', 'unknown']].to_numpy()
    ]),
    correctness=df['correct'].to_numpy(),
    num_chains=2
)
mcmc.print_summary()

In [ ]:
posterior_data = part2.analysis.posterior_to_arviz(
    mcmc,
    system_names=[name for name, _ in sorted(systems.items(), key=lambda pair: pair[1])],
    feature_names=feature_names
)

plot_data = posterior_data.copy()
plot_data["posterior"] = posterior_data.posterior["w"].transpose("chain", "draw", "feature", "system").to_dataset()

az.plot_forest(
    plot_data,
    var_names=["w"],
    combined=True,
    shade_label="feature",
    figure_kwargs={"figsize": (16, 8), "width_ratios": [2, 3], "layout": "none"},
    point_estimate='median', ci_kind='hdi', ci_probs=(0.5, 0.95),
)

# Final test on SST-2 validation (LSTM fine-tuned, GloVe 50d)

In [ ]:
sst2_train = list(zip(sst2['train']['sentence'], sst2['train']['label']))
sst2_val = list(zip(sst2['validation']['sentence'], sst2['validation']['label']))

metrics, best_epoch = part2.experiment.train(
    training_set=sst2_train,
    validation_set=sst2_val,
    tokenizer=tokenizer_for_rnn,
    emb_model=glove,
    num_classes=len(sst2_outcome_space),
    save_to="results/tmp-lstm-ft",
    enc_type='lstm',
    enc_kwargs={'freeze_embeddings': False, 'hidden_size': 20, 'word_dropout': 0.},
    regressor_kwargs={'hidden_size1': 50, 'hidden_size2': 50, 'dropout': 0.1},
    opt_kwargs={'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64, 'num_epochs': 8},
    device='cpu'
)

#Statistics for the best epoch
best_epoch_metrics = metrics[best_epoch]
print(f"Best epoch: {best_epoch}")
print(f"Train accuracy: {best_epoch_metrics['train_metrics']['accuracy']}")
print(f"Validation accuracy: {best_epoch_metrics['val_metrics']['accuracy']}")